# Download a list of CSDA STAC items by ID

Downloads specific CSDA holdings named by their STAC item id (here, 19 Planet `PSScene` scenes from June–July 2024 over Railroad Valley, NV). No search is needed: each item is read directly from the CSDA STAC (`/collections/<collection>/items/<id>`), then its assets are downloaded through the CSDA download API with your Earthdata login (`multistac_download`, the same login flow as `csda_client`).

| Step | What it does |
|---|---|
| 1. Configuration | Item ids, collection, which assets, output folder |
| 2. Get the items | Reads each item from CSDA STAC; lists any id that isn't found |
| 3. Review | Table and footprint map of the items; asset keys available |
| 4. Plan | Dry run: every file that would be downloaded, and where |
| 5. Download | Fetches the plan (only if `DRY_RUN = False`); existing files are skipped, so re-running resumes |
| 6. Check | Status per file; failed files to retry |

Output: `OUT_DIR/csda/<collection>/<item id>/<file>`, plus `download_manifest.csv` and the item JSON (`stac_items.json`) in `OUT_DIR`.

In [ ]:
import sys, json
from pathlib import Path
from types import SimpleNamespace
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

sys.path.insert(0, '/home/pmontesa/code/csda_summaries/lib')
import multistac
import multistac_download as md

## 1. Configuration — the only cell to edit

In [ ]:
ITEM_IDS = """
PSScene-20240604_184045_71_2486
PSScene-20240613_175955_44_2465
PSScene-20240614_183941_78_24a4
PSScene-20240615_174832_70_2417
PSScene-20240615_184227_20_2498
PSScene-20240615_184229_06_2498
PSScene-20240617_174628_36_24bf
PSScene-20240630_183824_74_24fc
PSScene-20240701_184437_63_2484
PSScene-20240704_174808_82_24b5
PSScene-20240705_175638_78_2456
PSScene-20240706_184718_75_2495
PSScene-20240708_174446_31_24a7
PSScene-20240709_175411_96_2415
PSScene-20240709_175112_15_24c9
PSScene-20240713_174917_81_24d0
PSScene-20240717_174720_61_24ca
PSScene-20240725_175011_58_2417
PSScene-20240730_175937_29_2423
""".split()

COLLECTION = 'planet'
STAC_URL   = multistac.load_profiles()['csda']['stac_url']     # https://csdap.earthdata.nasa.gov/stac

# Asset keys to download. None = every asset of each item. Planet PSScene items have:
#   basic_analytic_8b (+ _xml, basic_analytic_4b_rpc), basic_udm2, ortho_analytic_8b (+ _xml),
#   ortho_analytic_8b_sr, ortho_udm2, ortho_visual, json_metadata, thumbnail
ASSETS = None

OUT_DIR     = Path('/explore/nobackup/projects/CSDA_eval/csda_download/planet_railroad_valley_2024')
DRY_RUN     = True     # True: show the plan only. False: download in step 5.
MAX_WORKERS = 4

## 2. Get the items from CSDA STAC

In [ ]:
def get_item(collection, item_id):
    r = requests.get(f'{STAC_URL}/collections/{collection}/items/{item_id}', timeout=60)
    return r.json() if r.status_code == 200 else None

items, not_found = [], []
for i in dict.fromkeys(ITEM_IDS):              # keeps order, drops repeated ids
    it = get_item(COLLECTION, i)
    (items.append(it) if it else not_found.append(i))
print(f'{len(items)} of {len(set(ITEM_IDS))} items found in {COLLECTION}')
if not_found:
    print('NOT FOUND:', not_found)

In [ ]:
# One row per item; provider 'csda' routes downloads through the CSDA download API (stac_providers.yaml)
gdf = gpd.GeoDataFrame([{
        'provider': 'csda', 'collection': it['collection'], 'id': it['id'],
        'datetime': pd.to_datetime(it['properties'].get('datetime')),
        'item_type': it['properties'].get('pl:item_type'),
        'cloud_percent': it['properties'].get('pl:cloud_percent'),
        'n_assets': len(it['assets']),
        'geometry': shape(it['geometry'])} for it in items], crs='EPSG:4326')
res = SimpleNamespace(gdf=gdf, items=items)     # what multistac_download expects from a search result

OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / 'stac_items.json', 'w') as f:
    json.dump({'type': 'FeatureCollection', 'features': items}, f)
print('item JSON saved:', OUT_DIR / 'stac_items.json')

## 3. Review the items

In [ ]:
gdf.drop(columns='geometry').sort_values('datetime').reset_index(drop=True)

In [ ]:
gdf.assign(datetime=gdf.datetime.astype(str)).explore(tiles='Esri.WorldGrayCanvas', style_kwds={'fillOpacity': 0.05},
                                                      tooltip=['id', 'datetime', 'cloud_percent'])

In [ ]:
md.list_assets(res)

## 4. Plan (dry run)

In [ ]:
asset_keys = ASSETS if ASSETS is not None else sorted(md.asset_table(res)['asset'].unique())
plan = md.download(res, OUT_DIR, assets=asset_keys, dry_run=True)
print(f"{len(plan)} files for {plan['id'].nunique()} items | assets: {', '.join(asset_keys)}")
print('already on disk:', sum(Path(p).exists() and Path(p).stat().st_size > 0 for p in plan['path']))
plan[['id', 'asset', 'filename', 'path', 'status']].head(15)

## 5. Download (only if `DRY_RUN = False`)
Files that already exist are skipped, so re-running this cell resumes an interrupted download.

In [ ]:
if DRY_RUN:
    print(f'DRY_RUN: would download {len(plan)} files. Set DRY_RUN = False in section 1 and re-run.')
else:
    manifest = md.download(res, OUT_DIR, assets=asset_keys, max_workers=MAX_WORKERS)
    print('manifest:', OUT_DIR / 'download_manifest.csv')

## 6. Check

In [ ]:
manifest = pd.read_csv(OUT_DIR / 'download_manifest.csv') if (OUT_DIR / 'download_manifest.csv').exists() else plan
summary = manifest.groupby('status').agg(files=('path', 'size'), GB=('bytes', lambda b: round(b.sum() / 1e9, 2)))
display(summary)
failed = manifest[manifest.status == 'failed']
if len(failed):
    display(failed[['id', 'asset', 'error']])
    print('Re-run step 5 to retry the failed files.')

In [ ]:
# Files per item on disk
pd.Series({i: len(list((OUT_DIR / 'csda' / COLLECTION / i).glob('*'))) if (OUT_DIR / 'csda' / COLLECTION / i).exists() else 0
           for i in gdf.id}, name='files_on_disk').to_frame()